# Multi-Layer Perceptron (MLP)

This model is a neural network written from scratch in plain Python lists. It predicts the policy class of each *region* (`INCREASE = 1`, `RETAIN = 0`, `DECREASE = -1`) from its standardized features.

Architecture: **input → hidden (ReLU) → output (softmax, 3 classes)**, trained with batch gradient descent.

In [10]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import csv
import random
from collections import Counter
from statistics import mean, stdev
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path
from os import path

from methods.read_data import createMatrix, printMatrix
from methods.normalize_data import standardizeData, coalesceData
from models.mlp import gradient_descent, make_predictions, get_accuracy
from evaluation.holdout_validation import holdout_split, holdout_validation
from evaluation.kfold_cross_validation import kFoldSplit, crossValidation, accuracy
from evaluation.conf_matrix import evaluate

### Step 1: Create the Dataset from the `PROJECT_DATASET` folder
- This will create a matrix with 19 fields/attributes and 17 regions

In [12]:
data_set_dir = path.join(path.abspath("."), "PROJECT_DATASET")          # Get the path to the dataset folder
regions, fields, matrix = createMatrix(Path(data_set_dir))              # Create the matrix from the dataset
printMatrix(regions, fields, matrix, max_regions=18, max_fields=20)     # Print the matrix with a maximum of 18 regions and 20 fields

17 regions x 19 fields
showing fields 0 to 18 

  [0] Table_2 | Poverty Incidence among Population (%) 2023
  [1] Table_2 | Poverty Incidence among Population (%) 2021
  [2] Table_2 | Poverty Incidence among Population (%) 2018
  [3] Table_12 | Estimate (%) 2023
  [4] Table_6a | Estimate ('000) 2023
  [5] Table_2 | Annual Per Capita Poverty Threshold (in PhP) 2023
  [6] Total Family Size | Average Income
  [7] Total Family Size | Average Expenditure
  [8] Getting information about goods or services Percentage
  [9] Getting information from general government organizations Percentage
  [10] Interacting with government organizations Percentage
  [11] Seeking health-related information Percentage
  [12] Making an appointment with a health practitioner via the internet Percentage
  [13] Attending an online class/online course (on any subject) Percentage
  [14] online encyclopedias Percentage
  [15] websites for a job Percentage
  [16] DERIVED | Change in poverty incidence 2021 to 2023 (per

### Step 2: Cleanup the data
- Coalesce missing values to 0, then standardize it using Z-score Scaling
- `featuresList` is a list of feature vectors, in the same order as `regions`

In [13]:
# Coalesce NA values to 0, then standardize it.
cleaned_data = standardizeData(coalesceData(matrix))
featuresList = [list(row) for row in cleaned_data]      # Plain lists, so the label can be appended later

print(f"{len(regions)} regions, {len(featuresList[0])} features each")

17 regions, 19 features each


### Step 3: Load the labels from `region_policy_results.csv`
- Unlike k-means, the MLP is *supervised*, so every region needs its true label
- Read the actual class of each region from the tagged dataset, the same way as in the k-means notebook
- The class is converted to `INCREASE = 1`, `RETAIN = 0`, `DECREASE = -1`
- `labelList` follows the order of `regions`, so `featuresList[i]` and `labelList[i]` belong to the same region

In [14]:
# Read the actual class of each region from the tagged dataset
tagged_dir = path.join(data_set_dir, "region_policy_results.csv")      # Get the path to the tagged dataset
with open(tagged_dir, newline="", encoding="utf-8-sig") as file:
  actual_classes = {row["Region"]: row["Class"] for row in csv.DictReader(file)}

# Convert the class of each region to a number, in the same order as regions
CLASS_TO_LABEL = {"INCREASE": 1, "RETAIN": 0, "DECREASE": -1}
labelList = [CLASS_TO_LABEL[actual_classes[id]] for id in regions]

print("Label counts:", Counter(labelList))

Label counts: Counter({1: 9, 0: 6, -1: 2})


### Step 4: Map the labels to class indices
- The output layer has 3 neurons (one per class), so the network works with the indices `0, 1, 2`, not `-1, 0, 1`
- `CLASSES` is the single source of truth: `index[label]` going in, `CLASSES[i]` coming out
- A raw `-1` would silently land in the last column through negative indexing, so always map first

In [15]:
CLASSES = (-1, 0, 1)                                          # Column order of the output layer
index = {label: i for i, label in enumerate(CLASSES)}         # {-1: 0, 0: 1, 1: 2}
print("Label -> index:", index)

Label -> index: {-1: 0, 0: 1, 1: 2}


### Step 5: Split the regions into a training set and a testing set
- 80% of the regions are used for training and the remaining 20% for testing
- The training rows carry the true label as the last element, so it is stripped off before training

In [16]:
training_list, testing_rows, test_labels, test_indices = holdout_split(featuresList, labelList, 0.8)

X_train = [row[:-1] for row in training_list]                 # Features only
Y_train = [index[row[-1]] for row in training_list]           # Labels as class indices
print(f"Training regions: {len(X_train)}, Testing regions: {len(testing_rows)}")

Training regions: 14, Testing regions: 3


### Step 6: Train the network
- Hidden layer of 10 neurons, learning rate `alpha = 0.1`
- Each iteration runs one forward pass over all training regions, one backward pass (averaged gradients), and **one** weight update
- Training progress is printed every 2 iterations, so this demo uses only 20 iterations

In [17]:
random.seed(1)                                                # Same starting weights every run
w1, b1, w2, b2 = gradient_descent(X_train, Y_train, alpha=0.1, iterations=20, hiddenNum=10, outputNum=3)

Iteration:  0
Accuracy: 21.428571428571427%
Iteration:  2
Accuracy: 50.0%
Iteration:  4
Accuracy: 64.28571428571429%
Iteration:  6
Accuracy: 71.42857142857143%
Iteration:  8
Accuracy: 78.57142857142857%
Iteration:  10
Accuracy: 78.57142857142857%
Iteration:  12
Accuracy: 78.57142857142857%
Iteration:  14
Accuracy: 78.57142857142857%
Iteration:  16
Accuracy: 78.57142857142857%
Iteration:  18
Accuracy: 85.71428571428571%


### Step 7: Predict the regions in the testing set
- `make_predictions` returns class indices, which are decoded back to `-1 / 0 / 1` with `CLASSES`

In [18]:
predictions = [CLASSES[p] for p in make_predictions(testing_rows, w1, b1, w2, b2)]

print(f"{'Region':<25} {'True':>5} {'Predicted':>10}")
for i, true, pred in zip(test_indices, test_labels, predictions):
    print(f"{regions[i]:<25} {true:>5} {pred:>10}")

correct = sum(t == p for t, p in zip(test_labels, predictions))
print(f"\nTest accuracy: {correct}/{len(test_labels)} = {correct / len(test_labels):.2%}")

Region                     True  Predicted
BARMM                         1          1
Region I                      0          1
Region III                    0         -1

Test accuracy: 1/3 = 33.33%
